task : Clean duplicate records from the transaction dataset to retain only a single unique entry per transaction, ensuring accurate financial reporting.

In [0]:
transaction_df = spark.read.format("csv")\
    .option("Header","true")\
    .option("inferSchema","true")\
    .load("/Volumes/sql_problems/default/my_volume/day_07transaction.csv")

display(transaction_df)

In [0]:
from pyspark.sql.functions import *

filtered_df = transaction_df.filter(col("status") != "failed")

cleaned_df = transaction_df\
        .filter(col("status") == "success") \
        .dropDuplicates(["transaction_id", "customer_id"])

display(cleaned_df)


In [0]:
transaction_df.createOrReplaceTempView("transaction")

In [0]:
%sql
SELECT *,
ROW_NUMBER() OVER(
    PARTITION BY transaction_id, customer_id
    ORDER BY transaction_date
) AS row_number
FROM transaction
WHERE status = "success"
ORDER BY transaction_id, customer_id, row_number asc

In [0]:
%sql
WITH refined AS(
    SELECT *,
    ROW_NUMBER() OVER(
        PARTITION BY transaction_id, customer_id
        ORDER BY transaction_date
    ) AS row_number
    FROM transaction
    WHERE status = "success"
    ORDER BY transaction_id, customer_id, row_number asc
)
SELECT transaction_id, customer_id, transaction_date, amount, status
FROM refined
WHERE row_number = 1